In [1]:
import torch
from datasets import load_dataset
from transformers import GPT2Tokenizer

/home/bhuv/Documents/saidl_2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

In [3]:
dataset

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [4]:
for i in range(10):
    print(repr(dataset['train'][i]['text']))

''
' = Valkyria Chronicles III = \n'
''
' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'
" The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving 

In [5]:
lengths = [len(sample["text"]) for sample in dataset["train"]]
print(f"min: {min(lengths)}")
print(f"max: {max(lengths)}")
print(f"avg: {sum(lengths)/len(lengths):.1f}")
print(f"empty strings: {lengths.count(0)}")

min: 0
max: 3863
avg: 296.7
empty strings: 12951


In [6]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")


In [7]:
type(GPT2Tokenizer)

type

In [8]:
print(tokenizer.eos_token_id)

50256


In [9]:
tokens = []
for sentence in dataset["train"]:
    if sentence["text"].strip() != "":
        tokens.extend(tokenizer(sentence["text"])["input_ids"])
    else:
        continue
    tokens.append(tokenizer.eos_token_id)

In [18]:
len(tokenizer)

50257

In [10]:
tokens

[796,
 569,
 18354,
 7496,
 17740,
 6711,
 796,
 220,
 198,
 50256,
 2311,
 73,
 13090,
 645,
 569,
 18354,
 7496,
 513,
 1058,
 791,
 47398,
 17740,
 357,
 4960,
 1058,
 10545,
 230,
 99,
 161,
 254,
 112,
 5641,
 44444,
 9202,
 25084,
 24440,
 12675,
 11839,
 18,
 837,
 6578,
 764,
 569,
 18354,
 7496,
 286,
 262,
 30193,
 513,
 1267,
 837,
 8811,
 6412,
 284,
 355,
 569,
 18354,
 7496,
 17740,
 6711,
 2354,
 2869,
 837,
 318,
 257,
 16106,
 2597,
 2488,
 12,
 31,
 2712,
 2008,
 983,
 4166,
 416,
 29490,
 290,
 6343,
 13,
 44206,
 329,
 262,
 14047,
 44685,
 764,
 28728,
 287,
 3269,
 2813,
 287,
 2869,
 837,
 340,
 318,
 262,
 2368,
 983,
 287,
 262,
 569,
 18354,
 7496,
 2168,
 764,
 12645,
 278,
 262,
 976,
 21748,
 286,
 16106,
 290,
 1103,
 2488,
 12,
 31,
 640,
 11327,
 355,
 663,
 27677,
 837,
 262,
 1621,
 4539,
 10730,
 284,
 262,
 717,
 983,
 290,
 5679,
 262,
 366,
 17871,
 5321,
 366,
 837,
 257,
 23634,
 2422,
 4326,
 7351,
 262,
 3277,
 286,
 7096,
 544,
 1141,
 262,
 5

In [11]:
# packing and chunking
context_len = 1024
chunks = []

for i in range(0, len(tokens) - context_len, context_len):
    chunks.append(tokens[i:i+context_len])

In [12]:
torch.arange(0, 1024, dtype=torch.float).unsqueeze(1).size()

torch.Size([1024, 1])

In [13]:
torch.arange(0, 256, 2, dtype=torch.float).unsqueeze(0).size()

torch.Size([1, 128])

In [14]:
import torch.nn as nn
import torch as t

class BaselinePE(nn.Module):
    def __init__(self, dim: int, len: int = 1024):
        super().__init__()

        # blank canvas
        positional_encodings = t.zeros(len, dim)

        # positions and indices
        positions = t.arange(0, len, dtype=t.float).unsqueeze(1)
        even_idx = t.arange(0, dim, 2, dtype=t.float)
        

        # pre-calculating the denominator and its fractional value.
        denominator = 10000 ** (even_idx / dim)
        inv_denom = (1 / denominator).unsqueeze(0)

        # This was honestly crazy. Slicing through at intervals of two so that it fills only even rows.
        positional_encodings[:, 0::2] = t.sin(positions @ inv_denom)
        positional_encodings[:, 1::2] = t.cos(positions @ inv_denom)

        # Now since we have the (B, L, E) dimensions in mind.
        positional_encodings = positional_encodings.unsqueeze(0)

        # Registering as a buffer so it saves with the model but doesn't get trained
        self.register_buffer("positional_encodings", positional_encodings)

    def forward(self, x):
        x += self.positional_encodings
        return x


In [15]:
pe = BaselinePE(20, 10)

In [16]:
pe.positional_encodings.size()

torch.Size([1, 10, 20])

In [17]:
class SelfAttention(nn.Module):
    def __init__(self, ):
        super().__init__()